# **Install and Import Libraries**

> ##### **Make sure the secrets.env file is in the config folder. An example for secrets.env can be found in config/secrets_example.env file**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
import os

# load config
load_dotenv("../config/config.env")

# load secrets
load_dotenv("../config/secrets.env")

True

In [3]:
import requests
import feedparser
from neo4j import GraphDatabase
from datetime import datetime
from html.parser import HTMLParser
import json

# **1. Fetch and Parse RSS Feed**

In [4]:
RSS_FEED_URL = "https://www.malax.fi/nyheter/rss"

# Fetch the RSS feed
response = requests.get(RSS_FEED_URL)
feed = feedparser.parse(response.content)

print(f"Feed Title: {feed.feed.get('title', 'N/A')}")
print(f"Total items found: {len(feed.entries)}")
print(f"Feed link: {feed.feed.get('link', 'N/A')}")

Feed Title: Malax kommun - Nyheter
Total items found: 10
Feed link: https://www.malax.fi/


# **2. Extract and Clean News Data**

In [5]:
class HTMLStripper(HTMLParser):
    """Helper class to strip HTML tags from text"""
    def __init__(self):
        super().__init__()
        self.reset()
        self.strict = False
        self.convert_charrefs = True
        self.text = []
    
    def handle_data(self, d):
        self.text.append(d)
    
    def get_data(self):
        return ''.join(self.text).strip()

def strip_html(html_text):
    """Strip HTML tags from text"""
    if not html_text:
        return ""
    stripper = HTMLStripper()
    stripper.feed(html_text)
    return stripper.get_data()

def parse_date(date_string):
    """Parse RFC 2822 date format to YYYY-MM-DD format"""
    if not date_string:
        return None
    try:
        # feedparser already parses the date, convert to YYYY-MM-DD format
        dt = datetime(*date_string[:6])
        return dt.strftime("%Y-%m-%d")
    except:
        return None

def extract_first_sentences(text, num_sentences=2):
    """Extract first N sentences from text"""
    if not text:
        return ""
    # Split by period, question mark, or exclamation mark
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    result = ' '.join(sentences[:num_sentences])
    # Ensure it ends with proper punctuation
    if result and result[-1] not in '.!?':
        result += '.'
    return result

# Extract news items from Malax municipality feed
news_items = []
for entry in feed.entries:
    full_description = strip_html(entry.get("summary", entry.get("description", ""))).strip()
    news_item = {
        "title": entry.get("title", "").strip(),
        "description": extract_first_sentences(full_description, 2),
        "content": full_description,
        "link": entry.get("link", "").strip(),
        "publish_date": parse_date(entry.get("published_parsed")),
        "author": "",
        "source": "Malax",
        "image": "",
        "image_tag": ""
    }
    news_items.append(news_item)

print(f"Extracted {len(news_items)} news items")

print("\nFirst news item:")
print(json.dumps(news_items[0], indent=2, ensure_ascii=False))

Extracted 10 news items

First news item:
{
  "title": "Kom ihåg förbudet mot utomhushållning av fjäderfä 8.2–31.5.2026",
  "description": "För att förhindra spridning av fågelinfluensaviruset ska fjäderfä och andra fåglar skyddas från kontakt med vilda fåglar mellan 8.2 och 31.5. Fåglarna ska hållas antingen inomhus eller deras utevistelseområde ska vara helt inhägnat och täckt med ett tillräckligt tätt nät (25 mm hål) eller på annat liknande sätt.",
  "content": "För att förhindra spridning av fågelinfluensaviruset ska fjäderfä och andra fåglar skyddas från kontakt med vilda fåglar mellan 8.2 och 31.5. Fåglarna ska hållas antingen inomhus eller deras utevistelseområde ska vara helt inhägnat och täckt med ett tillräckligt tätt nät (25 mm hål) eller på annat liknande sätt.\nAnmälan om utomhushållning av fjäderfä eller andra fåglar:\nDen som håller fjäderfä eller andra fåglar utomhus mellan 8.2 och 31.5 ska göra en förhandsanmälan om utomhushållning av fjäderfä till kommunens djurveteri

In [6]:
# Display all extracted news items
for i, item in enumerate(news_items, 1):
    print(f"\n--- News Item {i} ---")
    print(f"Title: {item['title']}")
    print(f"Link: {item['link']}")
    print(f"Published: {item['publish_date']}")
    print(f"Description preview: {item['description'][:150]}...")


--- News Item 1 ---
Title: Kom ihåg förbudet mot utomhushållning av fjäderfä 8.2–31.5.2026
Link: https://www.malax.fi/nyheter/kom-ihag-forbudet-mot-utomhushallning-av-fjaderfa-8-231-5-2026
Published: 2026-03-06
Description preview: För att förhindra spridning av fågelinfluensaviruset ska fjäderfä och andra fåglar skyddas från kontakt med vilda fåglar mellan 8.2 och 31.5. Fåglarna...

--- News Item 2 ---
Title: Välkommen på pysanka‑målning! Ласкаво просимо на розпис писанок!
Link: https://www.malax.fi/nyheter/valkommen-pa-pysankamalning-laskavo-prosimo-na-rozpis-pisanok
Published: 2026-03-03
Description preview: Lördag 28.3.2026 kl. 13–17 ordnas ett kreativt och familjevänligt evenemang på Malakta (Töckmovägen 48), där vi tillsammans målar pysanka‑påskägg....

--- News Item 3 ---
Title: Föreningsbidrag
Link: https://www.malax.fi/nyheter/foreningsbidrag
Published: 2026-02-26
Description preview: Nu kan föreningar söka Malax kommuns bidrag för 2026! Vi beviljar bidrag för idrotts- och mo

# **1b. Fetch and Parse YLE RSS Feed (Malax News)**

In [7]:
YLE_RSS_FEED_URL = "https://yle.fi/rss/t/18-182375/sv"

# Fetch the YLE RSS feed
response_yle = requests.get(YLE_RSS_FEED_URL)
feed_yle = feedparser.parse(response_yle.content)

print(f"YLE Feed Title: {feed_yle.feed.get('title', 'N/A')}")
print(f"Total items found: {len(feed_yle.entries)}")
print(f"Feed link: {feed_yle.feed.get('link', 'N/A')}")

YLE Feed Title: Svenska Yle | Malax
Total items found: 20
Feed link: https://svenska.yle.fi


In [8]:
# Extract news items from YLE feed
yle_news_items = []
for entry in feed_yle.entries:
    full_description = strip_html(entry.get("summary", entry.get("description", ""))).strip()
    news_item = {
        "title": entry.get("title", "").strip(),
        "description": full_description,  # Use RSS summary as-is (already concise)
        "content": "",  # Will be populated by scraper
        "link": entry.get("link", "").strip(),
        "publish_date": parse_date(entry.get("published_parsed")),
        "author": "",  # Will be populated by scraper
        "source": "Yle",
        "image": "",  # Will be populated by scraper
        "image_tag": ""  # Will be populated by scraper
    }
    yle_news_items.append(news_item)

print(f"Extracted {len(yle_news_items)} YLE news items")

# Merge with existing news items (remove duplicates based on link)
existing_links = {item["link"] for item in news_items}
merged_news_items = news_items.copy()

for item in yle_news_items:
    if item["link"] not in existing_links:
        merged_news_items.append(item)
        existing_links.add(item["link"])

news_items = merged_news_items

print(f"\n--- Merged Results ---")
print(f"Original Malax feed items: {len(news_items) - len(yle_news_items)}")
print(f"YLE feed items: {len(yle_news_items)}")
print(f"New items from YLE: {len([item for item in yle_news_items if item['link'] not in {i['link'] for i in news_items[:len(news_items)-len(yle_news_items)]}])}")
print(f"Total merged items: {len(news_items)}")
print(f"\nFirst 3 merged news items:")
for i, item in enumerate(news_items[:3], 1):

    print(f"\n{i}. {item['title']}")    
    print(f"   Published: {item['publish_date']}")

Extracted 20 YLE news items

--- Merged Results ---
Original Malax feed items: 10
YLE feed items: 20
New items from YLE: 20
Total merged items: 30

First 3 merged news items:

1. Kom ihåg förbudet mot utomhushållning av fjäderfä 8.2–31.5.2026
   Published: 2026-03-06

2. Välkommen på pysanka‑målning! Ласкаво просимо на розпис писанок!
   Published: 2026-03-03

3. Föreningsbidrag
   Published: 2026-02-26


# **2b. Scrape Full Article Content from YLE Articles**

In [9]:
from bs4 import BeautifulSoup
import time
from urllib.parse import urlparse

# Rate limiting setup
REQUEST_DELAY = 1  # seconds between requests
MAX_ARTICLES_TO_SCRAPE = 20
REQUEST_TIMEOUT = 10  # seconds

In [10]:
# Test article analysis
test_url = "https://yle.fi/a/7-10095082"

print(f"Analyzing article structure from: {test_url}\n")

try:
    response = requests.get(test_url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Try to find article content - YLE uses various selectors
    # Looking for main content area
    content_candidates = [
        soup.find('article'),
        soup.find('div', {'class': 'article-content'}),
        soup.find('div', {'class': 'yle-article-content'}),
        soup.find('div', {'data-component': 'ArticleBody'}),
    ]
    
    main_content = next((c for c in content_candidates if c), None)
    
    # Extract title
    title = None
    title_candidates = [
        soup.find('h1'),
        soup.find('h1', {'class': 'article-headline'}),
    ]
    title_tag = next((t for t in title_candidates if t), None)
    if title_tag:
        title = title_tag.get_text(strip=True)
    
    # Extract author
    author = None
    author_candidates = [
        soup.find('span', {'class': 'author'}),
        soup.find('div', {'class': 'article-author'}),
        soup.find('div', {'data-component': 'ArticleAuthor'}),
    ]
    author_tag = next((a for a in author_candidates if a), None)
    if author_tag:
        author = author_tag.get_text(strip=True)
    
    # Extract publish date
    pub_date = None
    date_candidates = [
        soup.find('time'),
        soup.find('div', {'class': 'publish-date'}),
    ]
    date_tag = next((d for d in date_candidates if d), None)
    if date_tag:
        pub_date = date_tag.get('datetime') or date_tag.get_text(strip=True)
    
    print("=" * 80)
    print(f"Title found: {title is not None}")
    if title:
        print(f"  Content: {title[:100]}")
    print(f"\nAuthor found: {author is not None}")
    if author:
        print(f"  Content: {author[:100]}")
    print(f"\nPublish date found: {pub_date is not None}")
    if pub_date:
        print(f"  Content: {pub_date}")
    print(f"\nMain content container found: {main_content is not None}")
    if main_content:
        # Get paragraphs from content
        paragraphs = main_content.find_all('p')
        print(f"  Paragraphs found: {len(paragraphs)}")
        if paragraphs:
            text_preview = ' '.join(p.get_text(strip=True) for p in paragraphs[:3])
            print(f"  Text preview: {text_preview[:150]}...")
    print("=" * 80)
    
except requests.Timeout:
    print(f"✗ Request timeout after {REQUEST_TIMEOUT} seconds")
except requests.ConnectionError as e:
    print(f"✗ Connection error: {e}")
except Exception as e:
    print(f"✗ Error analyzing article: {e}")

Analyzing article structure from: https://yle.fi/a/7-10095082

Title found: True
  Content: Sjöbevakningen: Isarna väldigt oberäkneliga

Author found: False

Publish date found: True
  Content: 2026-03-15T12:18:40+02:00

Main content container found: True
  Paragraphs found: 4
  Text preview: I lördags räddades ett par personer från isen i Bottenviken och sjöbevakningen varnar nu för att alls röra sig på isen. Två personer hade tagit sig ut...


In [11]:
def scrape_yle_article(url):
    """
    Scrape full article content from YLE article URL.
    Returns dict with: content, author, publish_date, image, image_tag, or None on failure
    """
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract main content - try multiple selectors
        main_content = None
        content_selectors = [
            ('article', {}),
            ('div', {'class': 'article-content'}),
            ('div', {'data-component': 'ArticleBody'}),
            ('main', {}),
        ]
        
        for tag, attrs in content_selectors:
            main_content = soup.find(tag, attrs)
            if main_content:
                break
        
        if not main_content:
            return None
        
        # Extract all paragraphs and combine text
        paragraphs = main_content.find_all('p')
        if not paragraphs:
            return None
        
        content_text = '\n\n'.join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))
        
        # Extract author
        author = None
        
        # Try to find by aria-label first (YLE uses "Gjord av" - "Made by")
        author_tag = soup.find(attrs={'aria-label': 'Gjord av'})
        if author_tag:
            author = author_tag.get_text(strip=True)
        
        # Fallback to other selectors if aria-label didn't work
        if not author:
            author_selectors = [
                ('span', {'class': 'author'}),
                ('div', {'class': 'article-author'}),
                ('span', {'data-component': 'ArticleAuthor'}),
            ]
            
            for tag, attrs in author_selectors:
                author_tag = soup.find(tag, attrs)
                if author_tag:
                    author = author_tag.get_text(strip=True)
                    break
        
        # Extract publish date
        pub_date = None
        time_tag = soup.find('time')
        if time_tag:
            pub_date = time_tag.get('datetime')
            if pub_date and 'T' in pub_date:
                # Convert ISO format to YYYY-MM-DD
                pub_date = pub_date.split('T')[0]
        
        # Extract image URL and description (from figcaption)
        image_url = None
        image_description = None
        
        # Try to find first image in the article content
        img_tag = main_content.find('img')
        if img_tag:
            image_url = img_tag.get('src')
            # Look for figcaption in parent figure element or nearby
            figure = img_tag.find_parent('figure')
            if figure:
                figcaption = figure.find('figcaption')
                if figcaption:
                    image_description = figcaption.get_text(strip=True)
            # If no figcaption found, try to find figcaption as sibling
            if not image_description:
                figcaption = img_tag.find_next('figcaption')
                if figcaption:
                    image_description = figcaption.get_text(strip=True)
        
        return {
            'content': content_text,
            'author': author,
            'publish_date': pub_date,
            'image': image_url,
            'image_tag': image_description
        }
        
    except requests.Timeout:
        return None
    except requests.ConnectionError:
        return None
    except Exception:
        return None

def scrape_malax_article(url):
    """
    Scrape full article content from Malax municipality article URL.
    Returns dict with: image, or None on failure
    Note: Malax articles don't have image captions, so image_tag is left empty
    """
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract image URL (no caption for Malax articles)
        image_url = None
        
        # Try to find first image on the page that has an actual image URL (not data URI)
        all_images = soup.find_all('img')
        
        # Debug: Print all found images and their src attributes
        print(f"\n  DEBUG: Found {len(all_images)} images on page")
        for idx, img in enumerate(all_images[:5]):  # Show first 5
            src = img.get('src', 'NO SRC')
            data_src = img.get('data-src', 'NO DATA-SRC')
            print(f"    Image {idx}: src={src[:80] if src else 'None'}...")
            if data_src != 'NO DATA-SRC':
                print(f"             data-src={data_src[:80]}...")
        
        for img_tag in all_images:
            src = img_tag.get('src', '')
            # Try data-src attribute first (for lazy-loaded images)
            if not src or src.startswith('data:'):
                data_src = img_tag.get('data-src', '')
                if data_src and not data_src.startswith('data:'):
                    src = data_src
            
            # Skip placeholder/data URIs, look for actual image paths
            if src and not src.startswith('data:'):
                image_url = src
                print(f"  DEBUG: Selected image: {src[:80]}...")
                break
        
        # Convert relative URLs to absolute if needed
        if image_url and image_url.startswith('/'):
            image_url = 'https://www.malax.fi' + image_url
        
        return {
            'image': image_url,
            'image_tag': ''  # Malax doesn't have image captions
        }
        
    except requests.Timeout:
        return None
    except requests.ConnectionError:
        return None
    except Exception as e:
        print(f"  DEBUG: Exception in scrape_malax_article: {e}")
        return None

# Scrape YLE articles and Malax article images
print(f"Scraping full content from YLE articles (max {MAX_ARTICLES_TO_SCRAPE})...\n")

yle_articles = [item for item in news_items if 'yle.fi' in item['link']][:MAX_ARTICLES_TO_SCRAPE]

scraped_count = 0
failed_count = 0

for i, article in enumerate(yle_articles, 1):
    print(f"[{i}/{len(yle_articles)}] Scraping YLE: {article['title'][:60]}...", end="")
    
    scraped_data = scrape_yle_article(article['link'])
    
    if scraped_data and scraped_data['content']:
        article['content'] = scraped_data['content']
        if scraped_data['author']:
            article['author'] = scraped_data['author']
        if scraped_data['publish_date']:
            article['publish_date'] = scraped_data['publish_date']
        if scraped_data.get('image'):
            article['image'] = scraped_data['image']
        if scraped_data.get('image_tag'):
            article['image_tag'] = scraped_data['image_tag']
        scraped_count += 1
        print(" ✓")
    else:
        failed_count += 1
        print(" ✗")
    
    # Rate limiting - wait between requests (except after last one)
    if i < len(yle_articles):
        time.sleep(REQUEST_DELAY)

print(f"\n--- YLE Scraping Summary ---")
print(f"Total YLE articles: {len(yle_articles)}")
print(f"Successfully scraped: {scraped_count}")
print(f"Failed: {failed_count}")

# Scrape images from Malax articles
print(f"\nScraping images from Malax articles...\n")

malax_articles = [item for item in news_items if 'malax.fi' in item['link']]

malax_scraped_count = 0
malax_failed_count = 0

for i, article in enumerate(malax_articles, 1):
    print(f"[{i}/{len(malax_articles)}] Scraping Malax: {article['title'][:60]}...", end="")
    
    scraped_data = scrape_malax_article(article['link'])
    
    if scraped_data and scraped_data.get('image'):
        article['image'] = scraped_data['image']
        article['image_tag'] = scraped_data.get('image_tag', '')
        malax_scraped_count += 1
        print(" ✓")
    else:
        malax_failed_count += 1
        print(" ✗")
    
    # Rate limiting - wait between requests (except after last one)
    if i < len(malax_articles):
        time.sleep(REQUEST_DELAY)

print(f"\n--- Malax Scraping Summary ---")
print(f"Total Malax articles: {len(malax_articles)}")
print(f"Successfully scraped: {malax_scraped_count}")
print(f"Failed: {malax_failed_count}")

# Show samples of scraped articles
if scraped_count > 0:
    yle_sample = next((a for a in yle_articles if a.get('image')), None)
    if yle_sample:
        print(f"\nSample YLE article with image:")
        print(f"Title: {yle_sample['title']}")
        print(f"Image URL: {yle_sample.get('image', 'N/A')}")
        print(f"Image caption: {yle_sample.get('image_tag', 'N/A')}")

if malax_scraped_count > 0:
    malax_sample = next((a for a in malax_articles if a.get('image')), None)
    if malax_sample:
        print(f"\nSample Malax article with image:")
        print(f"Title: {malax_sample['title']}")
        print(f"Image URL: {malax_sample.get('image', 'N/A')}")


Scraping full content from YLE articles (max 20)...

[1/20] Scraping YLE: Fem gripande livsöden tävlar om att bli Lyssnarnas sommarpra... ✓
[2/20] Scraping YLE: Sjöbevakningen: Isarna väldigt oberäkneliga... ✓
[3/20] Scraping YLE: 400 vargobservationer på sex veckor – Malaxborna lever mitt ... ✓
[4/20] Scraping YLE: Kvarlämnad fiskfångst upprör i Malax: ”Totalt bortkastat”... ✓
[5/20] Scraping YLE: Hotet mot Kulkuri fick Vasa stad att utreda alternativ för h... ✓
[6/20] Scraping YLE: Älg överkörd av två långtradare och en paketbil i Kronoby... ✓
[7/20] Scraping YLE: Som schack på isen med 20 kilo tunga pjäser – curling är vin... ✓
[8/20] Scraping YLE: Hundgården Kulkuri i Vasa undvek konkurs... ✓
[9/20] Scraping YLE: 200 liter bränsle läckte ut på parkering vid skola i Malax... ✓
[10/20] Scraping YLE: Skidåkarna suktar efter snö: ”Vi gör snö på julafton om det ... ✓
[11/20] Scraping YLE: Fackanställd lokalpolitiker tar Malax kommun till rätten eft... ✓
[12/20] Scraping YLE: Kommunen kö

# **3. Connect to Neo4j Database**

In [12]:
# Initialize Neo4j driver
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Test connection
try:
    with driver.session() as session:
        result = session.run("RETURN 1")
        print("✓ Successfully connected to Neo4j database")
except Exception as e:
    print(f"✗ Failed to connect to Neo4j: {e}")

✓ Successfully connected to Neo4j database


# **4. Create News Nodes in Neo4j**

In [13]:
def create_or_update_news_node(driver, news_item):
    """
    Create or update a News node in Neo4j.
    Uses the link as a unique identifier to avoid duplicates.
    """
    with driver.session() as session:
        query = """
        MERGE (n:News {link: $link})
        SET 
            n.title = $title,
            n.description = $description,
            n.publish_date = $publish_date,
            n.content = $content,
            n.author = $author,
            n.source = $source,
            n.image = $image,
            n.image_tag = $image_tag,
            n.created_at = datetime()
        RETURN n
        """
        
        result = session.run(
            query,
            link=news_item["link"],
            title=news_item["title"],
            description=news_item["description"],
            publish_date=news_item["publish_date"],
            content=news_item.get("content"),
            author=news_item.get("author"),
            source=news_item.get("source"),
            image=news_item.get("image"),
            image_tag=news_item.get("image_tag")
        )
        
        return result.single()

# Create News nodes for all items
created_count = 0
updated_count = 0

print("Creating News nodes in Neo4j...")
for news_item in news_items:
    try:
        result = create_or_update_news_node(driver, news_item)
        if result:
            created_count += 1
            print(f"✓ Created/Updated: {news_item['title'][:50]}...")
    except Exception as e:
        print(f"✗ Error creating news node: {e}")
        print(f"  Item: {news_item['title']}")

print(f"Total items processed: {len(news_items)}")
print(f"Successfully created/updated: {created_count}")

print(f"\n--- Summary ---")

Creating News nodes in Neo4j...
✓ Created/Updated: Kom ihåg förbudet mot utomhushållning av fjäderfä ...
✓ Created/Updated: Välkommen på pysanka‑målning! Ласкаво просимо на р...
✓ Created/Updated: Föreningsbidrag...
✓ Created/Updated: Ansökningar till förskola, morgon- och eftermiddag...
✓ Created/Updated: Affischutställningen i Spåren av Sankt Olof kan se...
✓ Created/Updated: Skol-PT podden...
✓ Created/Updated: Föreslå mottagare av utmärkelser för 2025...
✓ Created/Updated: Ny veterinärlag träder i kraft 1.1.2026...
✓ Created/Updated: I form för livet...
✓ Created/Updated: Allmänna istider i Targahallen...
✓ Created/Updated: Fem gripande livsöden tävlar om att bli Lyssnarnas...
✓ Created/Updated: Sjöbevakningen: Isarna väldigt oberäkneliga...
✓ Created/Updated: 400 vargobservationer på sex veckor – Malaxborna l...
✓ Created/Updated: Kvarlämnad fiskfångst upprör i Malax: ”Totalt bort...
✓ Created/Updated: Hotet mot Kulkuri fick Vasa stad att utreda altern...
✓ Created/Updated: Älg öv

# **5. Verify News Nodes in Database**

In [14]:
def get_all_news_nodes(driver):
    """Retrieve all News nodes from the database ordered by publish date."""
    query = """
    MATCH (n:News)
    RETURN n.link as link, n.title as title, n.description as description, n.content as content, n.author as author, n.source as source, n.image as image, n.image_tag as image_tag, n.publish_date as publish_date, n.created_at as created_at
    ORDER BY n.publish_date DESC
    """
    with driver.session() as session:
        results = session.run(query)
        news_nodes = [dict(record) for record in results]
    return news_nodes

# Retrieve and display all News nodes
all_news = get_all_news_nodes(driver)
print(f"\n📰 All News Nodes in Database ({len(all_news)} total):\n")
for i, news in enumerate(all_news[:10], 1):
    print(f"{i}. {news['title']}")
    print(f"   Source: {news['source']}")
    print(f"   Link: {news['link']}")
    print(f"   Author: {news['author'] or 'N/A'}")
    print(f"   Publish Date: {news['publish_date']}")
    if news['image']:
        print(f"   Image URL: {news['image']}")
        if news['image_tag']:
            print(f"   Image Description: {news['image_tag']}")

    if news['content']:        print(f"   Content length: {len(news['content'])} chars")


📰 All News Nodes in Database (33 total):

1. Fem gripande livsöden tävlar om att bli Lyssnarnas sommarpratare
   Source: Yle
   Link: https://yle.fi/a/7-10095119?origin=rss
   Author: Fredrika Lindholm
   Publish Date: 2026-03-19
   Image URL: https://images.cdn.yle.fi/image/upload/c_crop,h_1080,w_1919,x_0,y_0/ar_1.7777777777777777,c_fill,g_faces,h_431,w_767/dpr_1.0/q_auto:eco/f_auto/fl_lossy/v1773750185/39-161287869b946dec3bee
   Image Description: Uppe till vänster Elin Kinnunen och bredvid henne Linus Söderlund. Nere till vänster Mikaela Blomqvist-Lyytikäinen, Cale Andersson och Jessica Hemming.Bild: Bilder från privata album. Grafiker: Paulina Nickström
   Content length: 1831 chars
2. Sjöbevakningen: Isarna väldigt oberäkneliga
   Source: Yle
   Link: https://yle.fi/a/7-10095082?origin=rss
   Author: Joni Kyheröinen
   Publish Date: 2026-03-15
   Image URL: https://images.cdn.yle.fi/image/upload/c_crop,h_785,w_1396,x_221,y_0/ar_1.7777777777777777,c_fill,g_faces,h_431,w_767/dpr_1.

# **6. Database Statistics**

In [15]:
def get_news_statistics(driver):
    """Get statistics about News nodes in the database."""
    with driver.session() as session:
        # Total count
        count_result = session.run("MATCH (n:News) RETURN count(n) as total")
        total_count = count_result.single()["total"]
        
        # Latest publish date
        latest_result = session.run("MATCH (n:News) RETURN max(n.publish_date) as latest")
        latest_date = latest_result.single()["latest"]
        
        # Oldest publish date
        oldest_result = session.run("MATCH (n:News) RETURN min(n.publish_date) as oldest")
        oldest_date = oldest_result.single()["oldest"]
    
    return {
        "total_count": total_count,
        "latest_date": latest_date,
        "oldest_date": oldest_date
    }

# Get and display statistics
stats = get_news_statistics(driver)
print(f"\n📊 Database Statistics:\n")
print(f"Total News Nodes: {stats['total_count']}")
print(f"Latest Article Date: {stats['latest_date']}")
print(f"Oldest Article Date: {stats['oldest_date']}")


📊 Database Statistics:

Total News Nodes: 33
Latest Article Date: 2026-03-19
Oldest Article Date: 2025-09-17


# **7. Close Database Connection**

In [16]:
driver.close()
print("✅ Database connection closed successfully!")

✅ Database connection closed successfully!
